## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following

- Tracking agent behaviour with logging ,analytics, and debugging .
- Transforming prompts,tool selection,and output formatting.
- Adding retries,fallbacks,and early termination logic.
- Appling rate limits,guardrails,and Pll detection

## Summarization Middleware

Automatically summarize conversation history  when approaching token limits,
preserving recent messages while compressing older context .

Summaerization is useful for the following:
- Long-running conversation that exceed context  windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
print("API key loaded:", bool(os.environ.get("GROQ_API_KEY")))

API key loaded: True


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:openai/gpt-oss-120b",  # Main agent model
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",  # Summarization model
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [17]:
### Run the thread id
config={"configurable":{"thread_id":"test-1"}}

In [ ]:

print("API key starts with:", os.environ.get("GROQ_API_KEY", "")[:5])

API key loaded: True
API key starts with: gsk_y


In [18]:
# Ailternative test data 
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?",
]

for q in questions:
   response=agent.invoke({"messages":[HumanMessage(content=q)]},config) 
   print(f"Messages:{response}")
   print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='543ee6a3-dd41-4ec2-bfa5-7ac7681e4cf2'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks a simple math question. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 78, 'total_tokens': 110, 'completion_time': 0.068166638, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.003118774, 'prompt_tokens_details': None, 'queue_time': 0.369598594, 'total_time': 0.071285412}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1f2c5-36f9-7196-abee-1153d27fcb63-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 32, 'total_tokens': 110, 'output_token_details': {'reasoning': 13}})]}
Messages: 2
Messag

### Token Size

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search  hotels - return long response to use  more tokens."""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star,$350/night,spa,pool,gym
    2.City Inn - 4 star,$180/night,bussiness center
    3.Budget Stay-3 star,$75/night,free wifi"""


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id":"hotel-test-clean-01"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return  total_chars//4 # 4 char = 1 token

In [27]:
# Run test 
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}:~{tokens}tokens,{len(response['messages'])}messages")
    print(f"{(response['messages'])}")

Paris:~662tokens,4messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='89872841-3396-4a46-b202-69c874fdf97a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We need to call the search_hotels function with city "Paris".', 'tool_calls': [{'id': 'fc_0894468c-d241-4378-a7bf-d774bb57e872', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 131, 'total_tokens': 183, 'completion_time': 0.107919183, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.006785513, 'prompt_tokens_details': None, 'queue_time': 0.153362267, 'total_time': 0.114704696}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1f2fe-e44f-7

## Fraction

In [36]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search  hotels - return long response to use  more tokens."""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star,$350/night,spa,pool,gym
    2.City Inn - 4 star,$180/night,bussiness center
    3.Budget Stay-3 star,$75/night,free wifi"""


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
     system_prompt="""
    You are a hotel search assistant.

    Rules:
    1. Always use the search_hotels tool when asked to find hotels.
    2. Only provide hotels returned by the tool.
    3. Never invent hotel names, prices, ratings, or amenities.
    4. Do not add information that the tool did not provide.
    5. If information is missing, clearly say it was not provided.
    """,
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b", 
            trigger=("fraction",0.005),#0.5%=~640 tokens   based llm context  size 
            keep=("fraction",0.002),#0.2%=~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id":"test-clean-02"}}
# Token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return  total_chars//4 # 4 char = 1 token

In [37]:
# Run test 
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens/131072
    print(f"{city}: ~{tokens}tokens({fraction:.4%}),{len(response['messages'])}msgs")
    print(response['messages'])

Paris: ~171tokens(0.1305%),4msgs
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='37c7559d-8cb2-4841-9af4-d7177056c1f6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_5bc414cb-9b30-4dbb-b411-4703fbc0f5b7', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 222, 'total_tokens': 263, 'completion_time': 0.086993195, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.025157748, 'prompt_tokens_details': None, 'queue_time': 0.158108851, 'total_time': 0.112150943}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1f31b-2687-705a-a0b6-c3dd3e1c3a5a-0', tool_calls=[{'name': 

## Human in the Loop MiddleWare

Pause age  execution for human approval,edditing,or rejection of tools calls before they execute.

Human-in-the loop is useful  for the  following:  

- High-stakes operations requiring Human approval(e.g. datbase writies,financial transactions).
- Compliance workfllows where  human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email contnent for ID:{email_id}"

def send_email_tool(recepient:str, subject:str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [40]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent=create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approved","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [43]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="send email to jhon@test.com with the subject 'hello' and body'how are you?'")]},
    config=config
)

In [44]:
result

{'messages': [HumanMessage(content="send email to jhon@test.com with the subject 'hello' and body'how are you?'", additional_kwargs={}, response_metadata={}, id='7eb6c68a-3378-4314-afaa-3ba4556f3b70'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to jhon@test.com with subject 'hello' and body 'how are you?'. The system has a function send_email_tool. We should use it. We need to format arguments: email_id? No, for sending email: body, recepient, subject. We'll call send_email_tool with those arguments.", 'tool_calls': [{'id': 'fc_86cbf3d0-0305-4b6c-a8dc-19c56c61f938', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 178, 'total_tokens': 290, 'completion_time': 0.116715049, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.008683287, 'pro

In [ ]:
config = {"configurable": {"thread_id": "reject-test-01"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to jhon@test.com with subject hello and body how are you?"
            )
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content='Send an email to jhon@test.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='3e5c3554-3ca2-4da3-8cc4-378dfa42b8aa'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function send_email_tool.', 'tool_calls': [{'id': 'fc_56d8a56f-cd9e-4b01-b3ec-92358d6b4a8f', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 175, 'total_tokens': 225, 'completion_time': 0.07235947, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.011769784, 'prompt_tokens_details': None, 'queue_time': 0.366873234, 'total_time': 0.084129254}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}

In [74]:
from langgraph.types import Command
# Step2: Approve
if "_interrupt__" in result:
    print("paused! Approving . . .")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ), config=config
    )

    print(f"Result:{result['messages'][-1].content}")

In [75]:
result

{'messages': [HumanMessage(content='Send an email to jhon@test.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='dced76fc-be06-49ce-a3b8-413da0179343'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email. There's a function send_email_tool. We should call that.", 'tool_calls': [{'id': 'fc_28cba55b-6e9f-4ef4-a63d-77fb3d277f66', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 175, 'total_tokens': 236, 'completion_time': 0.080485594, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.011751826, 'prompt_tokens_details': None, 'queue_time': 0.284320661, 'total_time': 0.09223742}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_

### Reject

In [90]:
def send_email_tool(recepient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recepient} with subject '{subject}' and body '{body}'"

In [91]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approved", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [78]:
from langgraph.types import Command

# Step 2: Reject
if "__interrupt__" in result:
    print("Paused! Rejecting email...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")
else:
    print("No interruption found!")

Paused! Rejecting email...
Result: I’m sorry, but I can’t send the email directly. However, I can draft it for you and you can copy it into your email client. Here’s the message:

**To:** jhon@test.com  
**Subject:** hello  

```
how are you?
```

Let me know if you’d like any changes or additional details!


In [79]:
result

{'messages': [HumanMessage(content="send email to jhon@test.com with the subject 'hello' and body'how are you?'", additional_kwargs={}, response_metadata={}, id='507db3bf-6caf-4b79-a2d7-d7be7fbd5974'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send an email using the send_email_tool. The user wants to send to jhon@test.com subject \'hello\' body \'how are you?\'. Note: There\'s a small typo: body\'how are you?\'. Probably body: "how are you?" We\'ll call the function.', 'tool_calls': [{'id': 'fc_d3a6a3ed-1458-47f6-ba1a-5ba9593a2fc5', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 178, 'total_tokens': 279, 'completion_time': 0.109819737, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.010314588, 'prompt_tokens_details': None, 'queue_time': 0.255060599,

## Edit

In [96]:
from langchain_core.tools import tool

In [97]:
@tool(return_direct=True)
def send_email_tool(recepient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recepient} with subject '{subject}' and body '{body}'"

In [98]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approved", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [99]:
config = {
    "configurable": {
        "thread_id": "test-edit-direct"
    }
}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to jhon@test.com with subject 'hello' and body 'how are you?'"
            )
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='48785461-447d-4a10-a909-1e9b7725d7d3'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use send_email_tool.', 'tool_calls': [{'id': 'fc_7f23615d-ba41-4085-8df2-bc341040d347', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 177, 'total_tokens': 226, 'completion_time': 0.052745556, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009235465, 'prompt_tokens_details': None, 'queue_time': 0.282808745, 'total_time': 0.061981021}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc

In [100]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Editing email...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recepient": "jhon@test.com",
                                "subject": "Updated Hello",
                                "body": "Hello John, this is my edited email!"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print("Final result:")
    print(result["messages"][-1].content)

else:
    print("No interruption found!")

Paused! Editing email...
Final result:
Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: send_email_tool with arguments {"recepient": "jhon@test.com", "subject": "Updated Hello", "body": "Hello John, this is my edited email!"}.

Tool response:
Email sent to jhon@test.com with subject 'Updated Hello' and body 'Hello John, this is my edited email!'


In [101]:
result

{'messages': [HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='48785461-447d-4a10-a909-1e9b7725d7d3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use send_email_tool.', 'tool_calls': [{'id': 'fc_7f23615d-ba41-4085-8df2-bc341040d347', 'function': {'arguments': '{"body":"how are you?","recepient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 177, 'total_tokens': 226, 'completion_time': 0.052745556, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009235465, 'prompt_tokens_details': None, 'queue_time': 0.282808745, 'total_time': 0.061981021}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='